In [19]:
import pandas as pd
import os
import glob
from ukhls_variables import VARIABLE_MAP

# --- SETTINGS ---
SIPHER_PKL = "../data/sipher/pickles/sipher_optimized.pkl"
PICKLE_DIR = "../data/ukhls/pickles/"

# Add generated variables that aren't in the base mapping but get created in intake
EXTENDED_MAP = VARIABLE_MAP.copy()
EXTENDED_MAP['primary_disability'] = 'Type of Impairment (Collapsed)'

def validate_integrity():
    print("--- Integrity Check: Low Data Presence (< 50%) ---")
    
    if not os.path.exists(SIPHER_PKL):
        print("Error: SIPHER Map pickle not found. Please run the Sipher Intake first.")
        return

    # 1. Load Sipher PIDP reference once to save time
    df_sipher = pd.read_pickle(SIPHER_PKL)[['pidp']]
    total_sipher_rows = len(df_sipher)
    sipher_pid_series = df_sipher['pidp']
    
    # 2. Find all pickle files
    pkl_files = sorted(glob.glob(os.path.join(PICKLE_DIR, "*.pkl")))
    if not pkl_files:
        print(f"No pickle files found in {PICKLE_DIR}.")
        return

    # To track visibility across ALL files
    globally_missing = {k: True for k in EXTENDED_MAP.keys() if k != 'pidp'}

    for pkl_file in pkl_files:
        filename = os.path.basename(pkl_file)
        print(f"\nAnalyzing File: {filename}")
        
        try:
            # We don't filter columns on load here because we want to adapt to whatever is inside
            df_master = pd.read_pickle(pkl_file)
        except Exception as e:
            print(f"  Error loading {filename}: {e}")
            continue

        if 'pidp' not in df_master.columns:
            print(f"  Skipping {filename}: 'pidp' column not found.")
            continue

        # Basic Match Analytics
        master_ids = set(df_master['pidp'].dropna().unique())
        matched_rows_count = sipher_pid_series.isin(master_ids).sum()

        print(f"  Total rows in file: {len(df_master):,}")
        print(f"  Matched rows with SIPHER Map: {matched_rows_count:,} ({(matched_rows_count / total_sipher_rows) * 100:.2f}%)")

        report_rows = []
        
        # Determine likely wave prefix from the filename (e.g., 'o_indresp' -> 'o_')
        likely_prefix = filename.split('_')[0] + "_" 
        
        for base_col, label in EXTENDED_MAP.items():
            if base_col == 'pidp':
                continue
                
            # Smart logic to find the column despite the file's wave
            target_col = None
            if f"{likely_prefix}{base_col}" in df_master.columns:
                target_col = f"{likely_prefix}{base_col}"
            elif f"o_{base_col}" in df_master.columns: # fallback if it's strictly expanded to master wave O
                target_col = f"o_{base_col}"
            elif base_col in df_master.columns: # fallback for generic/unprefixed columns
                target_col = base_col
            else:
                # Catch-all search for a column ending with the un-prefixed name
                matches = [c for c in df_master.columns if c.endswith(f"_{base_col}")]
                if matches:
                    target_col = matches[0]
            
            if target_col:
                globally_missing[base_col] = False # We found it in at least one file!
                null_count = int(df_master[target_col].isnull().sum())
                available = int(df_master[target_col].notnull().sum())
                valid_feature_ids = set(df_master.loc[df_master[target_col].notnull(), 'pidp'])
            else:
                null_count = "N/A"
                available = 0
                valid_feature_ids = set()
                target_col = f"Missing ({base_col})"
                    
            fill_rate = (available / len(df_master)) * 100 if len(df_master) else 0
            
            # --- ONLY REPORT VARIABLES WITH < 50% PRESENCE ---
            if fill_rate < 50.0:
                populated_rows = int(sipher_pid_series.isin(valid_feature_ids).sum())
                map_fill_rate = (populated_rows / total_sipher_rows) * 100 if total_sipher_rows else 0
                
                report_rows.append({
                    'Feature': label,
                    'Column Found': target_col,
                    'Missing Data': null_count,
                    'Available Data': available,
                    'Fill Rate %': round(fill_rate, 2),
                    'SIPHER Map %': round(map_fill_rate, 2),
                })

        if report_rows:
            feature_report = pd.DataFrame(report_rows)
            feature_report = feature_report.sort_values('Fill Rate %', ascending=False).reset_index(drop=True)
            print(f"  => Variables mapped with < 50% presence:")
            print(feature_report.to_string(index=False))
        else:
            print(f"  => Success: All dictionary variables have >= 50% presence in this file!")
            
        print("-" * 75)

    # --- FINAL MISSING VARIABLES SUMMARY ---
    missing_anywhere = [
        {'Base Variable': base, 'Feature Description': EXTENDED_MAP[base]} 
        for base, is_missing in globally_missing.items() if is_missing
    ]

    print("\n" + "=" * 75)
    print("FINAL SUMMARY: COMPLETELY MISSING VARIABLES")
    print("=" * 75)
    if missing_anywhere:
        print("The following variables were not found in ANY of the processed pickle files:\n")
        missing_df = pd.DataFrame(missing_anywhere)
        print(missing_df.to_string(index=False))
    else:
        print("Success! All dictionary variables were found in at least one file.")

validate_integrity()

--- Integrity Check: Low Data Presence (< 50%) ---

Analyzing File: l_indresp_optimized.pkl
  Total rows in file: 29,271
  Matched rows with SIPHER Map: 43,424,471 (82.16%)
  => Variables mapped with < 50% presence:
                                                     Feature         Column Found Missing Data  Available Data  Fill Rate %  SIPHER Map %
                       Monthly Net Pay (Total take-home pay)            l_payn_dv        17164           12107        41.36         35.66
                             One-way commute time in minutes    Missing (commute)          N/A               0         0.00          0.00
              Commute Mode (1=Drive, 3=Train, 4=Bus, 8=Walk)      Missing (jbmth)          N/A               0         0.00          0.00
                           WFH Frequency (1=Always, 5=Never)      Missing (home7)          N/A               0         0.00          0.00
       Public Transport Ease (1=Very Easy, 5=Very Difficult)   Missing (traccess)          N/A